# **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

if IS_COLAB:
    !pip install optuna

Cloning into 'RecSys-Challenge-2025'...
remote: Enumerating objects: 627, done.
remote: Counting objects: 100% (170/170), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 627 (delta 98), reused 126 (delta 63), pack-reused 457 (from 1)
Receiving objects: 100% (627/627), 22.34 MiB | 18.60 MiB/s, done.
Resolving deltas: 100% (318/318), done.


In [2]:
if IS_COLAB or True:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

run_compile_all_cython: Found 11 Cython files in 5 folders...
run_compile_all_cython: All files will be compiled using your current python environment: '/usr/bin/python3'
Compiling [1/11]: MatrixFactorizationImpressions_Cython_Epoch.pyx... 
In file included from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/ndarraytypes.h:1929,
                 from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/ndarrayobject.h:12,
                 from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/arrayobject.h:5,
                 from MatrixFactorizationImpressions_Cython_Epoch.c:1252:
/usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, disable it with " "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wcpp-Wcpp]8;;]
   17 | #warning "Using deprecated NumPy API, disable it with " \
 

In [3]:
import numpy as np
import optuna

from Challenge.paths import load_cv_folds
from Challenge.hyper_tuning import ModelOptimizer

Running on kaggle — storage at: /kaggle/working


/kaggle/working/RecSys-Challenge-2025/Challenge/hyper_tuning.py:75: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_study(self, study_name, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):
/kaggle/working/RecSys-Challenge-2025/Challenge/hyper_tuning.py:107: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_and_optimize_study(self, study_name, objective_function, n_trials=50, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):


# **Load Data**

In [4]:
# Load datasets
folds = load_cv_folds(k=5)

# **Hyperparameter search**

In [5]:
from Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython import MatrixFactorization_BPR_Cython

optimizer = ModelOptimizer("BPR")

STUDY_NAME = MatrixFactorization_BPR_Cython.RECOMMENDER_NAME

In [6]:
# Only One fold, too much time to train
URM_train, URM_val = folds[0]

from Evaluation.Evaluator import EvaluatorHoldout
evaluator = EvaluatorHoldout(URM_val, cutoff_list=[20])

early_stopping_params = {
    "validation_every_n": 5,
    "stop_on_validation": True,
    "evaluator_object": evaluator,
    "lower_validations_allowed": 5,
    "validation_metric": "RECALL",
    "epochs": 300
}

def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    reg_strength = optuna_trial.suggest_float("reg_strength", 1e-6, 1e-2, log=True)
    
    params = {
        "batch_size": 512,
        "sgd_mode": "adagrad",
        "learning_rate": optuna_trial.suggest_float("learning_rate", 1e-4, 5e-2, log=True),
        "num_factors": optuna_trial.suggest_int("num_factors", 32, 128, step=32),
        "user_reg": reg_strength,
        "positive_reg": reg_strength,
        "negative_reg": optuna_trial.suggest_float("negative_reg", 1e-6, 1e-3, log=True),
        "positive_threshold_BPR": None
    }
    
    recommender_instance = MatrixFactorization_BPR_Cython(URM_train)
    recommender_instance.fit(**params, **early_stopping_params)
    
    # Evaluate
    score = evaluator.evaluateRecommender(recommender_instance)[0].loc[20, "RECALL"]

    optimizer.log_folds([score], params)

    return score

EvaluatorHoldout: Ignoring 40 ( 0.1%) Users that have less than 1 test interactions


In [7]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=50
)

[I 2025-11-24 21:02:20,172] A new study created in RDB with name: MatrixFactorization_BPR_Cython_Recommender


  0%|          | 0/50 [00:00<?, ?it/s]

MF_BPR: Processed 27136 (100.0%) in 0.85 sec. MSE loss 1.92E-02. Sample per second: 32003
MF_BPR: Epoch 1 of 300. Elapsed time 0.38 sec
MF_BPR: Processed 27136 (100.0%) in 1.19 sec. MSE loss 1.95E-02. Sample per second: 22877
MF_BPR: Epoch 2 of 300. Elapsed time 0.72 sec
MF_BPR: Processed 27136 (100.0%) in 0.52 sec. MSE loss 1.94E-02. Sample per second: 51732
MF_BPR: Epoch 3 of 300. Elapsed time 1.06 sec
MF_BPR: Processed 27136 (100.0%) in 0.89 sec. MSE loss 1.93E-02. Sample per second: 30588
MF_BPR: Epoch 4 of 300. Elapsed time 1.42 sec
MF_BPR: Processed 27136 (100.0%) in 1.25 sec. MSE loss 1.94E-02. Sample per second: 21735
MF_BPR: Validation begins...
EvaluatorHoldout: Processed 27055 (100.0%) in 10.93 sec. Users per second: 2476
MF_BPR: CUTOFF: 20 - PRECISION: 0.0034448, PRECISION_RECALL_MIN_DEN: 0.0045086, RECALL: 0.0031062, MAP: 0.0006318, MAP_MIN_DEN: 0.0008223, MRR: 0.0116570, NDCG: 0.0039295, F1: 0.0032667, HIT_RATE: 0.0636112, ARHR_ALL_HITS: 0.0121110, NOVELTY: 0.0416222, AVE

In [8]:
optuna.visualization.plot_optimization_history(optuna_study)

In [9]:
optuna.visualization.plot_param_importances(optuna_study)

In [10]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Best Model**
- ADD HERE